In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import MaxAbsScaler

# ---------------------------------------------------------
# 1. DATA LOADING & INITIAL CLEANING
# ---------------------------------------------------------
# Using the target file: FOR_ML.csv
df = pd.read_csv('FOR_ML.csv')
df = df.dropna()

print("--- Step 1: Data Acquisition Complete ---")
print(f"Total patient records processed: {len(df):,}")

# ---------------------------------------------------------
# 2. FEATURE SELECTION & PREPROCESSING
# ---------------------------------------------------------
# Target is 'los'. Dropping 'id' to maintain clinical integrity.
y = df['los']
X = df.drop(columns=['los', 'id'], errors='ignore')

# Ensuring numerical consistency for clinical categories
categorical_cols = ['gender', 'age', 'ward', 'discipline', 'dis_shift', 'holiday_3', 'holiday_4', 'holiday_5']
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype(float)

scaler = MaxAbsScaler()
X_scaled = scaler.fit_transform(X)

print(f"--- Step 2: Feature Engineering Complete ({X.shape[1]} Features Scaled) ---")

# ---------------------------------------------------------
# 3. DATA PARTITIONING & MODEL TRAINING (Elastic Net)
# ---------------------------------------------------------
# Using the 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.20, 
    random_state=42
)

# Optimised hyperparameters
# Note: l1_ratio=1 means this effectively performed as a LASSO regression
model = ElasticNet(
    alpha=0.001,
    l1_ratio=1,
    max_iter=1000,
    selection='cyclic',
    tol=0.0001,
    random_state=0
)

print("Training the optimized Elastic Net model...")
model.fit(X_train, y_train)

print("--- Step 3: Model Development Complete ---")

# ---------------------------------------------------------
# 4. EVALUATION (TRAIN VS TEST)
# ---------------------------------------------------------
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("\n--- Step 4 & 5: Dissertation Results (Elastic Net Optimized) ---")

# Training Set Metrics
print(f"TRAIN - Mean Absolute Error (MAE): {mean_absolute_error(y_train, y_train_pred):.4f} days")
print(f"TRAIN - Root Mean Squared Error:    {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f} days")
print(f"TRAIN - R-squared (R2) Score:       {r2_score(y_train, y_train_pred):.4f}")

print("-" * 50)

# Testing Set Metrics
print(f"TEST  - Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_test_pred):.4f} days")
print(f"TEST  - Root Mean Squared Error:    {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f} days")
print(f"TEST  - R-squared (R2) Score:       {r2_score(y_test, y_test_pred):.4f}")

# ---------------------------------------------------------
# 5. COEFFICIENT ANALYSIS (Linear Predictor Impact)
# ---------------------------------------------------------
coef_df = pd.DataFrame({
    'Feature': [c for c in df.columns if c not in ['id', 'los']],
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("\nTop 15 Clinical Drivers of LOS (Elastic Net Coefficients):")
print(coef_df.head(15))